# Silver layer Analysis
In this first part, we do some analysis in order to get an understanding of how our bronze2silver transformations look like

In [0]:
%sql
-- describe let's you see the DDL of the table
describe ga_dev.bronze_metastore_osrs.ge_prices

In [0]:
%sql

select *
from ga_dev.bronze_metastore_osrs.ge_prices
limit 100;

# "Engineering" our Silver Layer and Tables
As you can see from the DDL and values in our bronze table, all our numbers are BIGINT. Notice how highTime and lowTime suggest time fields, but the values like unix timestamps. Also you see high and low values. 
We can create a silver table where it reads from the bronze, and write to the silver table and transform highTime and lowTime to timestamps. We will also create additional columns called "price_delta" and "time_delta" where and do high-low for cost-variance analysis and highTime-lowTime for duration analysis.

In [0]:
%sql
-- select query check on transformation logic
SELECT
  `id`,
  high,
  low,
  high-low as price_delta, 
  cast(from_unixtime(highTime) as timestamp) AS high_time_ts,
  cast(from_unixtime(lowTime) as timestamp) AS low_time_ts,
    ROUND((highTime - lowTime) / 3600.0,2) AS time_delta_hr --since it's in unixtimestamp in the original format, we can just divide by the total seconds by 3600 and round up to 2 for a time delta calculatoin
FROM ga_dev.bronze_metastore_osrs.ge_prices
ORDER BY high_time_ts DESC
LIMIT 100;

In [0]:
%sql

CREATE TABLE ga_dev.silver_metastore_osrs.ge_prices (
  id string,
  high decimal(18,2),
  low decimal(18,2),
  price_delta decimal(18,2),
  highTime timestamp,
  lowTime timestamp,
  time_delta_hr decimal(18,2),
  silver_load_timestamp timestamp -- I have this column to help keep track of when i wrote all this data to this table.
)
USING DELTA; -- delta format when writing to databricks sql warehouse; it is basically a parquet file with transaction logging and keeping track of table level meta-data. Look up ACID properties for data engineering.

# SQL, Pandas, PySpark... which one? (Totally not written with ChatGPT btw)

People often say, "Use whatever you're most comfortable with!" — and while comfort matters, there are solid technical reasons to choose one over the other depending on data scale and architecture.

## 🟦 SQL  
If your data is already  in tables and stored in a relational or Delta Lake format, SQL is typically the most lightweight and expressive option. It's efficient for joins, filters, and inserts, and avoids the overhead of loading data into memory. SQL is also accessible for both analysts and engineers.

## 🟨 Pandas  
Pandas is great for small to medium-sized datasets (generally < 5–10 GB), especially in exploratory analysis or one-off processing. However, because it operates in-memory, it's not well-suited for large-scale, production-grade pipelines. As data grows, memory constraints become a real bottleneck.

## 🟥 PySpark / Spark  
PySpark shines in large-scale, distributed environments. It can handle terabytes of data efficiently across a cluster. However, for small workloads, the overhead of initializing a Spark session and distributing tasks can outweigh its benefits. You may end up paying for cluster resources that aren't fully used.

If you're writing small batches of data, or doing lightweight processing, Spark may be overkill unless you're working in an environment that already runs on Spark (e.g., Databricks with Delta Lake). That said, Spark’s support for ACID transactions, schema evolution, and time travel through Delta Lake makes it a powerful tool for production-grade pipelines.

---

### 🤝 Final Advice
- Use **SQL** when working with structured, relational-style data already in tables.
- Use **Pandas** for quick manipulation of small-to-medium datasets in memory.
- Use **PySpark** when you need to scale out across big data or want advanced features like fault tolerance, schema enforcement, and distributed computing.

#### Current Use-case
- we use good ol trust SQL :^)


In [0]:
%sql

insert into ga_dev.silver_metastore_osrs.ge_prices
(
  `id`,
  high,
  low,
  price_delta,
  highTime,
  lowTime,
  time_delta_hr,
  silver_load_timestamp
)
SELECT
  `id`,
  high,
  low,
  high-low as price_delta,
  cast(from_unixtime(highTime) as timestamp) as highTime,
  cast(from_unixtime(lowTime) as timestamp) as lowTime,
  ROUND((highTime - lowTime) / 3600.0,2) AS time_delta_hr,
  current_timestamp
from ga_dev.bronze_metastore_osrs.ge_prices

In [0]:
%sql
-- check
select * 
from ga_dev.silver_metastore_osrs.ge_prices
limit 100;
